In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Audio Generator

This script generates participant-specific audio files for a Peripersonal Space (PPS) experiment.
It combines three audio segments:
1. Intro: Contains first 36 trials (18 inhale, 18 exhale)
2. Repetition Segment: Contains 22 trials (11 inhale, 11 exhale) - repeated as needed
3. Outro: Added at the end

For each participant, the script:
1. Reads their design CSV to determine trial count and timing
2. Calculates how many repetition segments are needed
3. Combines audio segments (intro + repetition segments + outro)
4. Injects looming stimuli into one track and tactile stimuli into another track
5. Saves the resulting audio files

Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import soundfile as sf
import math
import re
import logging
from pathlib import Path

# Set up logging
logging.basicConfig(level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s')

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
AUDIO_CONFIG = {
    'paths': {
        'intro': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_intro.wav",
        'repetition_segment': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_repetitionsegment.wav",
        'outro': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_outro.wav",
        'design_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
        'output_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio",
        'stimulus_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles",
    },
    'debug_mode': True,
    # Number of trials in intro and repetition segment
    'intro_trials': 36,     # 18 inhale, 18 exhale
    'rep_segment_trials': 22,  # 11 inhale, 11 exhale
    'fade_duration_ms': 50,  # Fade duration for smooth transitions
}

# Looming stimulus mapping (match stimulus_type in design CSV to file)
LOOMING_STIMULI = {
    'blue': 'front_az0_FABIAN_HRIR_blue.wav',
    'brown': 'front_az0_FABIAN_HRIR_brown.wav',
    'pink': 'front_az0_FABIAN_HRIR_pink.wav',
    'white': 'front_az0_FABIAN_HRIR_white.wav',
    # Include directional variants if needed
    'right': 'right_az90_FABIAN_HRIR_natural.wav',
    'front': 'front_az0_FABIAN_HRIR_natural.wav',
    'left': 'left_az-90_FABIAN_HRIR_natural.wav',
}

# Tactile stimulus file
TACTILE_STIMULUS = 'tactile_stimulus.wav'

# ----------------------------------------------------------------------
# Helper function for parsing timestamps
# ----------------------------------------------------------------------
def parse_timestamp(timestamp):
    """
    Parse timestamp in format MM:SS.S to milliseconds.
    
    Args:
        timestamp: String in format "MM:SS.S"
        
    Returns:
        Float: milliseconds
    """
    if pd.isna(timestamp):
        return None
        
    match = re.match(r'(\d+):(\d+\.\d+)', timestamp)
    if match:
        minutes, seconds = match.groups()
        return (int(minutes) * 60 + float(seconds)) * 1000
    return None

# ----------------------------------------------------------------------
# CLASS: PPSAudioGenerator
# ----------------------------------------------------------------------
class PPSAudioGenerator:
    """Generates participant-specific audio files for PPS experiments."""
    
    def __init__(self, config=None):
        self.config = config or AUDIO_CONFIG
        
        # Prepare output directory
        self.output_dir = self.config['paths']['output_dir']
        os.makedirs(self.output_dir, exist_ok=True)
        
        # Load base audio segments
        self.intro_audio, self.sample_rate = self._load_audio(self.config['paths']['intro'])
        self.rep_segment_audio, _ = self._load_audio(self.config['paths']['repetition_segment'])
        self.outro_audio, _ = self._load_audio(self.config['paths']['outro'])
        
        # Load stimulus files
        self.looming_stimuli = self._load_looming_stimuli()
        self.tactile_stimulus, _ = self._load_audio(os.path.join(
            self.config['paths']['stimulus_dir'], TACTILE_STIMULUS))
        
        if self.config['debug_mode']:
            self._log_audio_info()
    
    def _load_audio(self, file_path):
        """Load audio file and return data and sample rate."""
        try:
            data, sample_rate = sf.read(file_path)
            return data, sample_rate
        except Exception as e:
            logging.error(f"Error loading audio file {file_path}: {e}")
            raise
    
    def _load_looming_stimuli(self):
        """Load all looming stimulus files."""
        looming_stimuli = {}
        for key, filename in LOOMING_STIMULI.items():
            file_path = os.path.join(self.config['paths']['stimulus_dir'], filename)
            try:
                data, _ = self._load_audio(file_path)
                looming_stimuli[key] = data
                if self.config['debug_mode']:
                    logging.info(f"Loaded looming stimulus '{key}' from {filename}")
            except Exception as e:
                logging.warning(f"Could not load looming stimulus '{key}' from {filename}: {e}")
        return looming_stimuli
    
    def _log_audio_info(self):
        """Log information about loaded audio files for debugging."""
        logging.info(f"Loaded audio files:")
        intro_duration = len(self.intro_audio) / self.sample_rate
        rep_duration = len(self.rep_segment_audio) / self.sample_rate
        outro_duration = len(self.outro_audio) / self.sample_rate
        
        logging.info(f"  Intro: {intro_duration:.2f} seconds, {self.intro_audio.shape}")
        logging.info(f"  Repetition Segment: {rep_duration:.2f} seconds, {self.rep_segment_audio.shape}")
        logging.info(f"  Outro: {outro_duration:.2f} seconds, {self.outro_audio.shape}")
        logging.info(f"  Sample Rate: {self.sample_rate} Hz")
        
        for key, data in self.looming_stimuli.items():
            logging.info(f"  Looming '{key}': {len(data)/self.sample_rate:.2f} seconds, {data.shape}")
        
        logging.info(f"  Tactile: {len(self.tactile_stimulus)/self.sample_rate:.2f} seconds, {self.tactile_stimulus.shape}")
    
    def load_participant_design(self, participant_id):
        """
        Load participant design CSV and return the DataFrame.
        
        Args:
            participant_id: Participant ID number
            
        Returns:
            DataFrame containing the trial design
        """
        design_path = os.path.join(
            self.config['paths']['design_dir'], 
            f"participant_{participant_id}_design.csv")
        
        try:
            design_df = pd.read_csv(design_path)
            
            # Parse timestamp columns to milliseconds
            for col in ['looming_stimulus_timestamp', 'tactile_stimulus_timestamp']:
                if col in design_df.columns:
                    design_df[f'{col}_ms'] = design_df[col].apply(parse_timestamp)
            
            if self.config['debug_mode']:
                max_trial = design_df['trial_number'].max()
                logging.info(f"Loaded design for participant {participant_id}")
                logging.info(f"  Total trials: {len(design_df)}")
                logging.info(f"  Max trial number: {max_trial}")
                logging.info(f"  Trial types: {design_df['trial_type'].value_counts().to_dict()}")
            
            return design_df
            
        except Exception as e:
            logging.error(f"Error loading design for participant {participant_id}: {e}")
            raise
    
    def calculate_repetition_segments(self, design_df):
        """
        Calculate how many repetition segments are needed based on trial count.
        
        Args:
            design_df: DataFrame containing the trial design
            
        Returns:
            Integer: Number of repetition segments needed
        """
        max_trial = design_df['trial_number'].max()
        
        # Subtract intro trials, then divide by repetition segment trials
        remaining_trials = max(0, max_trial - self.config['intro_trials'])
        segments_needed = math.ceil(remaining_trials / self.config['rep_segment_trials'])
        
        if self.config['debug_mode']:
            logging.info(f"Calculated repetition segments:")
            logging.info(f"  Max trial: {max_trial}")
            logging.info(f"  Intro trials: {self.config['intro_trials']}")
            logging.info(f"  Remaining trials: {remaining_trials}")
            logging.info(f"  Trials per segment: {self.config['rep_segment_trials']}")
            logging.info(f"  Segments needed: {segments_needed}")
        
        return segments_needed
    
    def combine_audio_segments(self, num_segments):
        """
        Combine intro, repetition segments, and outro into a single audio stream.
        
        Args:
            num_segments: Number of repetition segments to include
            
        Returns:
            Numpy array: Combined audio data
        """
        # Get shapes and create fade arrays
        if len(self.intro_audio.shape) > 1:  # Stereo
            channels = self.intro_audio.shape[1]
            fade_samples = int(self.config['fade_duration_ms'] * self.sample_rate / 1000)
            fade_in = np.linspace(0, 1, fade_samples)[:, np.newaxis]
            fade_in = np.tile(fade_in, (1, channels))
            fade_out = np.linspace(1, 0, fade_samples)[:, np.newaxis]
            fade_out = np.tile(fade_out, (1, channels))
        else:  # Mono
            fade_samples = int(self.config['fade_duration_ms'] * self.sample_rate / 1000)
            fade_in = np.linspace(0, 1, fade_samples)
            fade_out = np.linspace(1, 0, fade_samples)
        
        # List to store audio segments
        segments = [self.intro_audio]
        
        # Add repetition segments
        for i in range(num_segments):
            segment = self.rep_segment_audio.copy()
            segments.append(segment)
        
        # Add outro
        segments.append(self.outro_audio)
        
        # Combine all segments
        combined_audio = np.concatenate(segments)
        
        if self.config['debug_mode']:
            total_duration = len(combined_audio) / self.sample_rate
            logging.info(f"Combined audio segments:")
            logging.info(f"  Total duration: {total_duration:.2f} seconds")
            logging.info(f"  Total samples: {len(combined_audio)}")
        
        return combined_audio
    
    def create_injection_map(self, design_df):
        """
        Create a map of where to inject looming and tactile stimuli.
        
        Args:
            design_df: DataFrame with the design
            
        Returns:
            Dictionary with injection details
        """
        injection_map = {
            'looming': [],
            'tactile': []
        }
        
        # Process each trial
        for _, row in design_df.iterrows():
            trial_type = row['trial_type']
            
            # Skip practice trials
            if trial_type == 'practice':
                continue
            
            # Looming stimuli for inhalation, exhalation, and catch trials
            if trial_type in ['inhalation', 'exhalation', 'catch'] and not pd.isna(row['looming_stimulus_timestamp_ms']):
                looming_ms = row['looming_stimulus_timestamp_ms']
                stim_type = row['stimulus_type']
                
                if stim_type in self.looming_stimuli:
                    injection_map['looming'].append({
                        'time_ms': looming_ms,
                        'stimulus_type': stim_type,
                        'trial_number': row['trial_number'],
                        'trial_type': trial_type
                    })
            
            # Tactile stimuli for inhalation, exhalation, and baseline trials
            if trial_type in ['inhalation', 'exhalation', 'baseline'] and not pd.isna(row['tactile_stimulus_timestamp_ms']):
                tactile_ms = row['tactile_stimulus_timestamp_ms']
                
                injection_map['tactile'].append({
                    'time_ms': tactile_ms,
                    'trial_number': row['trial_number'],
                    'trial_type': trial_type
                })
        
        # Sort injections by time
        injection_map['looming'].sort(key=lambda x: x['time_ms'])
        injection_map['tactile'].sort(key=lambda x: x['time_ms'])
        
        if self.config['debug_mode']:
            logging.info(f"Created injection map:")
            logging.info(f"  Looming injections: {len(injection_map['looming'])}")
            logging.info(f"  Tactile injections: {len(injection_map['tactile'])}")
        
        return injection_map
    
    def inject_stimuli(self, base_audio, injection_map):
        """
        Inject stimuli into the base audio according to the injection map.
        
        Args:
            base_audio: Base audio data
            injection_map: Dictionary with injection details
            
        Returns:
            Tuple of (looming_audio, tactile_audio)
        """
        # Create copies of base audio for each stimulus type
        looming_audio = base_audio.copy()
        tactile_audio = base_audio.copy()
        
        # Track the injections for verification
        injection_log = {
            'looming': [],
            'tactile': []
        }
        
        # Inject looming stimuli
        for inject in injection_map['looming']:
            time_ms = inject['time_ms']
            stimulus_type = inject['stimulus_type']
            trial_number = inject['trial_number']
            trial_type = inject['trial_type']
            
            # Convert time to samples
            start_sample = int(time_ms * self.sample_rate / 1000)
            
            # Get stimulus data
            stimulus_data = self.looming_stimuli.get(stimulus_type)
            if stimulus_data is None:
                logging.warning(f"No stimulus data for type '{stimulus_type}'")
                continue
                
            # Check if injection fits within audio bounds
            if start_sample + len(stimulus_data) <= len(looming_audio):
                # Handle stereo/mono differences
                if len(looming_audio.shape) > 1 and len(stimulus_data.shape) > 1:
                    # Both are stereo
                    looming_audio[start_sample:start_sample+len(stimulus_data)] += stimulus_data
                elif len(looming_audio.shape) > 1 and len(stimulus_data.shape) == 1:
                    # Looming is stereo, stimulus is mono
                    for ch in range(looming_audio.shape[1]):
                        looming_audio[start_sample:start_sample+len(stimulus_data), ch] += stimulus_data
                elif len(looming_audio.shape) == 1 and len(stimulus_data.shape) > 1:
                    # Looming is mono, stimulus is stereo
                    looming_audio[start_sample:start_sample+len(stimulus_data)] += np.mean(stimulus_data, axis=1)
                else:
                    # Both are mono
                    looming_audio[start_sample:start_sample+len(stimulus_data)] += stimulus_data
                
                # Log successful injection
                injection_log['looming'].append({
                    'trial_number': trial_number,
                    'trial_type': trial_type,
                    'stimulus_type': stimulus_type,
                    'time_ms': time_ms,
                    'time_sec': time_ms / 1000,
                    'sample_index': start_sample
                })
            else:
                logging.warning(f"Looming stimulus at {time_ms}ms (trial {trial_number}) exceeds audio length")
        
        # Inject tactile stimuli
        for inject in injection_map['tactile']:
            time_ms = inject['time_ms']
            trial_number = inject['trial_number']
            trial_type = inject['trial_type']
            
            # Convert time to samples
            start_sample = int(time_ms * self.sample_rate / 1000)
            
            # Check if injection fits within audio bounds
            if start_sample + len(self.tactile_stimulus) <= len(tactile_audio):
                # Handle stereo/mono differences
                if len(tactile_audio.shape) > 1 and len(self.tactile_stimulus.shape) > 1:
                    # Both are stereo
                    tactile_audio[start_sample:start_sample+len(self.tactile_stimulus)] += self.tactile_stimulus
                elif len(tactile_audio.shape) > 1 and len(self.tactile_stimulus.shape) == 1:
                    # Tactile audio is stereo, stimulus is mono
                    for ch in range(tactile_audio.shape[1]):
                        tactile_audio[start_sample:start_sample+len(self.tactile_stimulus), ch] += self.tactile_stimulus
                elif len(tactile_audio.shape) == 1 and len(self.tactile_stimulus.shape) > 1:
                    # Tactile audio is mono, stimulus is stereo
                    tactile_audio[start_sample:start_sample+len(self.tactile_stimulus)] += np.mean(self.tactile_stimulus, axis=1)
                else:
                    # Both are mono
                    tactile_audio[start_sample:start_sample+len(self.tactile_stimulus)] += self.tactile_stimulus
                
                # Log successful injection
                injection_log['tactile'].append({
                    'trial_number': trial_number,
                    'trial_type': trial_type,
                    'time_ms': time_ms,
                    'time_sec': time_ms / 1000,
                    'sample_index': start_sample
                })
            else:
                logging.warning(f"Tactile stimulus at {time_ms}ms (trial {trial_number}) exceeds audio length")
        
        if self.config['debug_mode']:
            logging.info(f"Injected stimuli:")
            logging.info(f"  Looming injections completed: {len(injection_log['looming'])}")
            logging.info(f"  Tactile injections completed: {len(injection_log['tactile'])}")
        
        return looming_audio, tactile_audio, injection_log
    
    def save_audio_files(self, participant_id, looming_audio, tactile_audio, injection_log):
        """
        Save the generated audio files and injection log.
        
        Args:
            participant_id: Participant ID
            looming_audio: Looming audio data
            tactile_audio: Tactile audio data
            injection_log: Dictionary with injection details
            
        Returns:
            Tuple of file paths (looming_path, tactile_path, log_path)
        """
        # Create filenames
        looming_filename = f"participant_{participant_id}_design_looming.wav"
        tactile_filename = f"participant_{participant_id}_design_tactile.wav"
        log_filename = f"participant_{participant_id}_design_stimuli_log.csv"
        
        # Create full paths
        looming_path = os.path.join(self.output_dir, looming_filename)
        tactile_path = os.path.join(self.output_dir, tactile_filename)
        log_path = os.path.join(self.output_dir, log_filename)
        
        # Save audio files
        sf.write(looming_path, looming_audio, self.sample_rate)
        sf.write(tactile_path, tactile_audio, self.sample_rate)
        
        # Save injection log as CSV
        # Combine looming and tactile injections
        all_injections = []
        
        for inject in injection_log['looming']:
            inject['stimulus'] = 'looming'
            all_injections.append(inject)
            
        for inject in injection_log['tactile']:
            inject['stimulus'] = 'tactile'
            # Add stimulus_type field for consistency
            inject['stimulus_type'] = 'tactile'
            all_injections.append(inject)
        
        # Convert to DataFrame and sort by time
        log_df = pd.DataFrame(all_injections)
        log_df = log_df.sort_values('time_ms').reset_index(drop=True)
        
        # Save to CSV
        log_df.to_csv(log_path, index=False)
        
        if self.config['debug_mode']:
            logging.info(f"Saved audio files for participant {participant_id}:")
            logging.info(f"  Looming: {looming_path}")
            logging.info(f"  Tactile: {tactile_path}")
            logging.info(f"  Log: {log_path}")
        
        return looming_path, tactile_path, log_path
    
    def verify_audio_sync(self, injection_log):
        """
        Verify audio synchronization by comparing expected SOA values against actual.
        
        Args:
            injection_log: Dictionary with injection details
            
        Returns:
            DataFrame with verification results
        """
        verification_results = []
        
        # Create dict for faster lookup
        looming_dict = {inject['trial_number']: inject for inject in injection_log['looming']}
        tactile_dict = {inject['trial_number']: inject for inject in injection_log['tactile']}
        
        # Check each looming trial that should have a tactile counterpart
        for trial_num, looming in looming_dict.items():
            if looming['trial_type'] == 'catch':
                continue  # Skip catch trials
                
            if trial_num in tactile_dict:
                tactile = tactile_dict[trial_num]
                
                actual_soa_ms = tactile['time_ms'] - looming['time_ms']
                
                verification_results.append({
                    'trial_number': trial_num,
                    'trial_type': looming['trial_type'],
                    'looming_time_ms': looming['time_ms'],
                    'tactile_time_ms': tactile['time_ms'],
                    'actual_soa_ms': actual_soa_ms
                })
        
        results_df = pd.DataFrame(verification_results)
        
        if self.config['debug_mode'] and not results_df.empty:
            # Print first few verification results
            logging.info(f"Verification sample (first 5 trials):")
            
            for _, row in results_df.head(5).iterrows():
                logging.info(f"  Trial {row['trial_number']} ({row['trial_type']}): " 
                            f"SOA = {row['actual_soa_ms']:.2f}ms")
            
            # Calculate statistics
            mean_soa = results_df['actual_soa_ms'].mean()
            min_soa = results_df['actual_soa_ms'].min()
            max_soa = results_df['actual_soa_ms'].max()
            
            logging.info(f"SOA statistics:")
            logging.info(f"  Mean: {mean_soa:.2f}ms")
            logging.info(f"  Min: {min_soa:.2f}ms")
            logging.info(f"  Max: {max_soa:.2f}ms")
        
        return results_df
    
    def process_participant(self, participant_id):
        """
        Process a single participant: load design, combine audio, inject stimuli, save files.
        
        Args:
            participant_id: Participant ID
            
        Returns:
            Tuple of (success, file_paths)
        """
        try:
            if self.config['debug_mode']:
                logging.info(f"\n=== Processing participant {participant_id} ===")
            
            # Load design CSV
            design_df = self.load_participant_design(participant_id)
            
            # Calculate repetition segments needed
            num_segments = self.calculate_repetition_segments(design_df)
            
            # Combine audio segments
            combined_audio = self.combine_audio_segments(num_segments)
            
            # Create injection map
            injection_map = self.create_injection_map(design_df)
            
            # Inject stimuli
            looming_audio, tactile_audio, injection_log = self.inject_stimuli(
                combined_audio, injection_map)
            
            # Verify synchronization
            verification = self.verify_audio_sync(injection_log)
            
            # Save audio files
            looming_path, tactile_path, log_path = self.save_audio_files(
                participant_id, looming_audio, tactile_audio, injection_log)
            
            return True, (looming_path, tactile_path, log_path)
            
        except Exception as e:
            logging.error(f"Error processing participant {participant_id}: {e}")
            import traceback
            traceback.print_exc()
            return False, None
    
    def run(self, participant_ids=None):
        """
        Run audio generation for multiple participants.
        
        Args:
            participant_ids: List of participant IDs or None for all
            
        Returns:
            Dictionary with results
        """
        # If no participant IDs provided, find all design CSVs
        if participant_ids is None:
            # Find all participant design files
            design_dir = self.config['paths']['design_dir']
            design_files = [f for f in os.listdir(design_dir) if f.startswith('participant_') and f.endswith('_design.csv')]
            
            # Extract participant IDs
            participant_ids = []
            for filename in design_files:
                match = re.match(r'participant_(\d+)_design\.csv', filename)
                if match:
                    participant_ids.append(int(match.group(1)))
            
            participant_ids.sort()  # Sort numerically
        
        if self.config['debug_mode']:
            logging.info(f"=== Starting PPSAudioGenerator ===")
            logging.info(f"Processing {len(participant_ids)} participants: {participant_ids}")
        
        results = {}
        for pid in participant_ids:
            success, paths = self.process_participant(pid)
            results[pid] = {'success': success, 'paths': paths}
        
        # Summary
        if self.config['debug_mode']:
            success_count = sum(1 for info in results.values() if info['success'])
            logging.info(f"\n=== GENERATION SUMMARY ===")
            logging.info(f"Successfully generated audio for {success_count}/{len(results)} participants.")
        
        return results

# ----------------------------------------------------------------------
# MAIN FUNCTION
# ----------------------------------------------------------------------
def main():
    """Main function to run the audio generator."""
    # Print configuration summary
    print("\n===== PPS EXPERIMENT AUDIO GENERATOR =====")
    print(f"Intro file: {AUDIO_CONFIG['paths']['intro']}")
    print(f"Repetition segment file: {AUDIO_CONFIG['paths']['repetition_segment']}")
    print(f"Outro file: {AUDIO_CONFIG['paths']['outro']}")
    print(f"Output directory: {AUDIO_CONFIG['paths']['output_dir']}")
    
    # Create the generator
    generator = PPSAudioGenerator(AUDIO_CONFIG)
    
    # Get participant IDs to process
    # You can specify specific IDs here or let it find all design files
    participant_ids = None  # Process all available participants
    # participant_ids = [0, 1, 2]  # Process specific participants
    
    # Run the generator
    results = generator.run(participant_ids)
    
    # Print summary
    print("\n===== GENERATION SUMMARY =====")
    success_count = sum(1 for info in results.values() if info['success'])
    print(f"Successfully generated audio for {success_count}/{len(results)} participants.")
    
    for pid, info in results.items():
        if info['success']:
            looming_path, tactile_path, log_path = info['paths']
            looming_file = os.path.basename(looming_path)
            tactile_file = os.path.basename(tactile_path)
            print(f"Participant {pid}: Files generated")
            print(f"  Looming: {looming_file}")
            print(f"  Tactile: {tactile_file}")
        else:
            print(f"Participant {pid}: ERROR - audio generation failed.")

if __name__ == "__main__":
    main()


===== PPS EXPERIMENT AUDIO GENERATOR =====
Intro file: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_intro.wav
Repetition segment file: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_repetitionsegment.wav
Outro file: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_outro.wav
Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio


2025-03-27 11:51:19,698 - INFO - Loaded looming stimulus 'blue' from front_az0_FABIAN_HRIR_blue.wav
2025-03-27 11:51:19,711 - INFO - Loaded looming stimulus 'brown' from front_az0_FABIAN_HRIR_brown.wav
2025-03-27 11:51:19,727 - INFO - Loaded looming stimulus 'pink' from front_az0_FABIAN_HRIR_pink.wav
2025-03-27 11:51:19,750 - INFO - Loaded looming stimulus 'white' from front_az0_FABIAN_HRIR_white.wav
2025-03-27 11:51:19,751 - ERROR - Error loading audio file C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\right_az90_FABIAN_HRIR_natural.wav: Error opening 'C:\\Users\\cogpsy-vrlab\\Documents\\GitHub\\BreathingSpace\\Level0_StimulusFiles\\right_az90_FABIAN_HRIR_natural.wav': System error.
2025-03-27 11:51:19,752 - WARNING - Could not load looming stimulus 'right' from right_az90_FABIAN_HRIR_natural.wav: Error opening 'C:\\Users\\cogpsy-vrlab\\Documents\\GitHub\\BreathingSpace\\Level0_StimulusFiles\\right_az90_FABIAN_HRIR_natural.wav': System error.
2025-03-27 11


===== GENERATION SUMMARY =====
Successfully generated audio for 100/100 participants.
Participant 0: Files generated
  Looming: participant_0_design_looming.wav
  Tactile: participant_0_design_tactile.wav
Participant 1: Files generated
  Looming: participant_1_design_looming.wav
  Tactile: participant_1_design_tactile.wav
Participant 2: Files generated
  Looming: participant_2_design_looming.wav
  Tactile: participant_2_design_tactile.wav
Participant 3: Files generated
  Looming: participant_3_design_looming.wav
  Tactile: participant_3_design_tactile.wav
Participant 4: Files generated
  Looming: participant_4_design_looming.wav
  Tactile: participant_4_design_tactile.wav
Participant 5: Files generated
  Looming: participant_5_design_looming.wav
  Tactile: participant_5_design_tactile.wav
Participant 6: Files generated
  Looming: participant_6_design_looming.wav
  Tactile: participant_6_design_tactile.wav
Participant 7: Files generated
  Looming: participant_7_design_looming.wav
  Tac

In [5]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
PPS Audio Generator (Revised)

This script generates participant-specific audio files for a Peripersonal Space (PPS) experiment.
It first creates an "empty audio" file by combining intro, repetition segments, and outro.
Then, for each participant, it overlays looming stimuli onto this empty audio based on their design CSV.

The script reads design CSVs from:
"C:\\Users\\cogpsy-vrlab\\Documents\\GitHub\\BreathingSpace\\Level1_AudioGeneration\\ExperimentLog"

The script looks for the following columns in the CSV:
- looming_stimulus_timestamp: Timestamp for looming stimulus
- tactile_stimulus_timestamp: Timestamp for tactile stimulus
- stimulus_type: Type of looming stimulus (pink, brown, white, blue)

For each participant, the script creates two files:
1. A looming audio file with the base audio and looming stimuli overlaid at specified timestamps
2. A tactile audio file with tactile stimuli at specified timestamps

Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import soundfile as sf
import math
import re
import logging
from pathlib import Path

# Set up logging
logging.basicConfig(level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s')

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
AUDIO_CONFIG = {
    'paths': {
        'intro': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_intro.wav",
        'repetition_segment': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_repetitionsegment.wav",
        'outro': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_outro.wav",
        'design_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
        'output_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio",
        'stimulus_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles",
    },
    'debug_mode': True,
    # Number of trials in intro and repetition segment
    'intro_trials': 36,     # 18 inhale, 18 exhale
    'rep_segment_trials': 22,  # 11 inhale, 11 exhale
    'fade_duration_ms': 50,  # Fade duration for smooth transitions
}

# Looming stimulus mapping
LOOMING_STIMULI = {
    'blue': 'frontal_looming_blue_noise.wav',
    'brown': 'frontal_looming_brown_noise.wav',
    'pink': 'frontal_looming_pink_noise.wav',
    'white': 'frontal_looming_white_noise.wav',
}

# Tactile stimulus file
TACTILE_STIMULUS = 'tactile_stimulus.wav'

# ----------------------------------------------------------------------
# Helper function for parsing timestamps
# ----------------------------------------------------------------------
def parse_timestamp(timestamp):
    """
    Parse timestamp in format MM:SS.S to milliseconds.
    
    Args:
        timestamp: String in format "MM:SS.S"
        
    Returns:
        Float: milliseconds
    """
    if pd.isna(timestamp):
        return None
        
    match = re.match(r'(\d+):(\d+\.\d+)', timestamp)
    if match:
        minutes, seconds = match.groups()
        return (int(minutes) * 60 + float(seconds)) * 1000
    return None

# ----------------------------------------------------------------------
# CLASS: PPSAudioGenerator
# ----------------------------------------------------------------------
class PPSAudioGenerator:
    """Generates participant-specific audio files for PPS experiments."""
    
    def __init__(self, config=None):
        self.config = config or AUDIO_CONFIG
        
        # Prepare output directory
        self.output_dir = self.config['paths']['output_dir']
        os.makedirs(self.output_dir, exist_ok=True)
        
        # Load base audio segments
        self.intro_audio, self.sample_rate = self._load_audio(self.config['paths']['intro'])
        self.rep_segment_audio, _ = self._load_audio(self.config['paths']['repetition_segment'])
        self.outro_audio, _ = self._load_audio(self.config['paths']['outro'])
        
        # Load stimulus files
        self.looming_stimuli = self._load_looming_stimuli()
        self.tactile_stimulus, _ = self._load_audio(os.path.join(
            self.config['paths']['stimulus_dir'], TACTILE_STIMULUS))
        
        # Empty audio cache
        self.empty_audio_cache = {}
        
        if self.config['debug_mode']:
            self._log_audio_info()
    
    def _load_audio(self, file_path):
        """Load audio file and return data and sample rate."""
        try:
            data, sample_rate = sf.read(file_path)
            return data, sample_rate
        except Exception as e:
            logging.error(f"Error loading audio file {file_path}: {e}")
            raise
    
    def _load_looming_stimuli(self):
        """Load all looming stimulus files."""
        looming_stimuli = {}
        for key, filename in LOOMING_STIMULI.items():
            file_path = os.path.join(self.config['paths']['stimulus_dir'], filename)
            try:
                data, _ = self._load_audio(file_path)
                looming_stimuli[key] = data
                if self.config['debug_mode']:
                    logging.info(f"Loaded looming stimulus '{key}' from {filename}")
            except Exception as e:
                logging.warning(f"Could not load looming stimulus '{key}' from {filename}: {e}")
        return looming_stimuli
    
    def _log_audio_info(self):
        """Log information about loaded audio files for debugging."""
        logging.info(f"Loaded audio files:")
        intro_duration = len(self.intro_audio) / self.sample_rate
        rep_duration = len(self.rep_segment_audio) / self.sample_rate
        outro_duration = len(self.outro_audio) / self.sample_rate
        
        logging.info(f"  Intro: {intro_duration:.2f} seconds, {self.intro_audio.shape}")
        logging.info(f"  Repetition Segment: {rep_duration:.2f} seconds, {self.rep_segment_audio.shape}")
        logging.info(f"  Outro: {outro_duration:.2f} seconds, {self.outro_audio.shape}")
        logging.info(f"  Sample Rate: {self.sample_rate} Hz")
        
        for key, data in self.looming_stimuli.items():
            logging.info(f"  Looming '{key}': {len(data)/self.sample_rate:.2f} seconds, {data.shape}")
        
        logging.info(f"  Tactile: {len(self.tactile_stimulus)/self.sample_rate:.2f} seconds, {self.tactile_stimulus.shape}")
    
    def find_max_trial_number(self):
        """
        Find the maximum trial number across all participant design CSVs.
        This helps determine the longest possible empty audio needed.
        
        Returns:
            Integer: Maximum trial number
        """
        max_trial = 0
        design_dir = self.config['paths']['design_dir']
        design_files = [f for f in os.listdir(design_dir) if f.startswith('participant_') and f.endswith('_design.csv')]
        
        for design_file in design_files:
            try:
                df = pd.read_csv(os.path.join(design_dir, design_file))
                if 'trial_number' in df.columns:
                    file_max = df['trial_number'].max()
                    max_trial = max(max_trial, file_max)
                    
            except Exception as e:
                logging.error(f"Error reading {design_file}: {e}")
        
        if self.config['debug_mode']:
            logging.info(f"Maximum trial number across all CSVs: {max_trial}")
        
        return max_trial
    
    def create_empty_audio(self, max_trial_number):
        """
        Create an empty audio file (intro + repetition segments + outro).
        
        Args:
            max_trial_number: Maximum trial number to accommodate
            
        Returns:
            Numpy array: Combined audio data
        """
        # Calculate how many repetition segments are needed
        remaining_trials = max(0, max_trial_number - self.config['intro_trials'])
        segments_needed = math.ceil(remaining_trials / self.config['rep_segment_trials'])
        
        if self.config['debug_mode']:
            logging.info(f"Creating empty audio for {max_trial_number} trials:")
            logging.info(f"  Intro trials: {self.config['intro_trials']}")
            logging.info(f"  Remaining trials: {remaining_trials}")
            logging.info(f"  Trials per segment: {self.config['rep_segment_trials']}")
            logging.info(f"  Repetition segments needed: {segments_needed}")
        
        # List to store audio segments
        segments = [self.intro_audio]
        
        # Add repetition segments
        for i in range(segments_needed):
            segment = self.rep_segment_audio.copy()
            segments.append(segment)
        
        # Add outro
        segments.append(self.outro_audio)
        
        # Combine all segments
        combined_audio = np.concatenate(segments)
        
        if self.config['debug_mode']:
            total_duration = len(combined_audio) / self.sample_rate
            logging.info(f"Created empty audio:")
            logging.info(f"  Total duration: {total_duration:.2f} seconds")
            logging.info(f"  Total samples: {len(combined_audio)}")
        
        return combined_audio
    
    def save_empty_audio(self, empty_audio, max_trial_number):
        """
        Save the empty audio file.
        
        Args:
            empty_audio: Empty audio data
            max_trial_number: Maximum trial number this audio accommodates
            
        Returns:
            String: Path to saved file
        """
        filename = f"empty_audio_trials_{max_trial_number}.wav"
        file_path = os.path.join(self.output_dir, filename)
        
        sf.write(file_path, empty_audio, self.sample_rate)
        
        if self.config['debug_mode']:
            logging.info(f"Saved empty audio to {file_path}")
        
        return file_path
    
    def get_empty_audio(self, max_trial_number):
        """
        Get empty audio, creating and caching if necessary.
        
        Args:
            max_trial_number: Maximum trial number needed
            
        Returns:
            Numpy array: Empty audio data
        """
        # Check if we already have suitable empty audio in cache
        for trials, audio in self.empty_audio_cache.items():
            if trials >= max_trial_number:
                if self.config['debug_mode']:
                    logging.info(f"Using cached empty audio for {trials} trials")
                return audio
        
        # Create new empty audio
        empty_audio = self.create_empty_audio(max_trial_number)
        
        # Save for reference
        self.save_empty_audio(empty_audio, max_trial_number)
        
        # Cache for future use
        self.empty_audio_cache[max_trial_number] = empty_audio
        
        return empty_audio
    
    def load_participant_design(self, participant_id):
        """
        Load participant design CSV and return the DataFrame.
        
        Args:
            participant_id: Participant ID number
            
        Returns:
            DataFrame containing the trial design
        """
        design_path = os.path.join(
            self.config['paths']['design_dir'], 
            f"participant_{participant_id}_design.csv")
        
        try:
            design_df = pd.read_csv(design_path)
            
            # Parse timestamp columns to milliseconds
            for col in ['looming_stimulus_timestamp', 'tactile_stimulus_timestamp']:
                if col in design_df.columns:
                    design_df[f'{col}_ms'] = design_df[col].apply(parse_timestamp)
            
            if self.config['debug_mode']:
                max_trial = design_df['trial_number'].max()
                logging.info(f"Loaded design for participant {participant_id}")
                logging.info(f"  Total trials: {len(design_df)}")
                logging.info(f"  Max trial number: {max_trial}")
                logging.info(f"  Trial types: {design_df['trial_type'].value_counts().to_dict()}")
            
            return design_df
            
        except Exception as e:
            logging.error(f"Error loading design for participant {participant_id}: {e}")
            raise
    
    def create_injection_map(self, design_df):
        """
        Create a map of where to inject looming and tactile stimuli.
        
        Args:
            design_df: DataFrame with the design
            
        Returns:
            Dictionary with injection details
        """
        injection_map = {
            'looming': [],
            'tactile': []
        }
        
        # Process each trial
        for _, row in design_df.iterrows():
            trial_type = row['trial_type']
            
            # Skip practice trials
            if trial_type == 'practice':
                continue
            
            # Looming stimuli for inhalation, exhalation, and catch trials
            if trial_type in ['inhalation', 'exhalation', 'catch'] and 'looming_stimulus_timestamp_ms' in row and not pd.isna(row['looming_stimulus_timestamp_ms']):
                looming_ms = row['looming_stimulus_timestamp_ms']
                stim_type = row['stimulus_type']
                
                if stim_type in self.looming_stimuli:
                    injection_map['looming'].append({
                        'time_ms': looming_ms,
                        'stimulus_type': stim_type,
                        'trial_number': row['trial_number'],
                        'trial_type': trial_type
                    })
            
            # Tactile stimuli for inhalation, exhalation, and baseline trials
            if trial_type in ['inhalation', 'exhalation', 'baseline'] and 'tactile_stimulus_timestamp_ms' in row and not pd.isna(row['tactile_stimulus_timestamp_ms']):
                tactile_ms = row['tactile_stimulus_timestamp_ms']
                
                injection_map['tactile'].append({
                    'time_ms': tactile_ms,
                    'trial_number': row['trial_number'],
                    'trial_type': trial_type
                })
        
        # Sort injections by time
        injection_map['looming'].sort(key=lambda x: x['time_ms'])
        injection_map['tactile'].sort(key=lambda x: x['time_ms'])
        
        if self.config['debug_mode']:
            logging.info(f"Created injection map:")
            logging.info(f"  Looming injections: {len(injection_map['looming'])}")
            logging.info(f"  Tactile injections: {len(injection_map['tactile'])}")
        
        return injection_map
    
    def create_participant_audio(self, participant_id, empty_audio, injection_map):
        """
        Create participant-specific audio by overlaying stimuli on empty audio.
        
        Args:
            participant_id: Participant ID
            empty_audio: Empty audio data
            injection_map: Dictionary with injection details
            
        Returns:
            Tuple of (looming_audio, tactile_audio, injection_log)
        """
        # Create copies of empty audio for each stimulus type
        looming_audio = empty_audio.copy()
        tactile_audio = np.zeros_like(empty_audio)  # Start with empty for tactile
        
        # Track the injections for verification
        injection_log = {
            'looming': [],
            'tactile': []
        }
        
        # Inject looming stimuli
        for inject in injection_map['looming']:
            time_ms = inject['time_ms']
            stimulus_type = inject['stimulus_type']
            trial_number = inject['trial_number']
            trial_type = inject['trial_type']
            
            # Convert time to samples
            start_sample = int(time_ms * self.sample_rate / 1000)
            
            # Get stimulus data
            stimulus_data = self.looming_stimuli.get(stimulus_type)
            if stimulus_data is None:
                logging.warning(f"No stimulus data for type '{stimulus_type}'")
                continue
                
            # Check if injection fits within audio bounds
            if start_sample + len(stimulus_data) <= len(looming_audio):
                # Handle stereo/mono differences
                if len(looming_audio.shape) > 1 and len(stimulus_data.shape) > 1:
                    # Both are stereo
                    looming_audio[start_sample:start_sample+len(stimulus_data)] += stimulus_data
                elif len(looming_audio.shape) > 1 and len(stimulus_data.shape) == 1:
                    # Looming is stereo, stimulus is mono
                    for ch in range(looming_audio.shape[1]):
                        looming_audio[start_sample:start_sample+len(stimulus_data), ch] += stimulus_data
                elif len(looming_audio.shape) == 1 and len(stimulus_data.shape) > 1:
                    # Looming is mono, stimulus is stereo
                    looming_audio[start_sample:start_sample+len(stimulus_data)] += np.mean(stimulus_data, axis=1)
                else:
                    # Both are mono
                    looming_audio[start_sample:start_sample+len(stimulus_data)] += stimulus_data
                
                # Log successful injection
                injection_log['looming'].append({
                    'trial_number': trial_number,
                    'trial_type': trial_type,
                    'stimulus_type': stimulus_type,
                    'time_ms': time_ms,
                    'time_sec': time_ms / 1000,
                    'sample_index': start_sample
                })
            else:
                logging.warning(f"Looming stimulus at {time_ms}ms (trial {trial_number}) exceeds audio length")
        
        # Inject tactile stimuli
        for inject in injection_map['tactile']:
            time_ms = inject['time_ms']
            trial_number = inject['trial_number']
            trial_type = inject['trial_type']
            
            # Convert time to samples
            start_sample = int(time_ms * self.sample_rate / 1000)
            
            # Check if injection fits within audio bounds
            if start_sample + len(self.tactile_stimulus) <= len(tactile_audio):
                # Handle stereo/mono differences
                if len(tactile_audio.shape) > 1 and len(self.tactile_stimulus.shape) > 1:
                    # Both are stereo
                    tactile_audio[start_sample:start_sample+len(self.tactile_stimulus)] += self.tactile_stimulus
                elif len(tactile_audio.shape) > 1 and len(self.tactile_stimulus.shape) == 1:
                    # Tactile audio is stereo, stimulus is mono
                    for ch in range(tactile_audio.shape[1]):
                        tactile_audio[start_sample:start_sample+len(self.tactile_stimulus), ch] += self.tactile_stimulus
                elif len(tactile_audio.shape) == 1 and len(self.tactile_stimulus.shape) > 1:
                    # Tactile audio is mono, stimulus is stereo
                    tactile_audio[start_sample:start_sample+len(self.tactile_stimulus)] += np.mean(self.tactile_stimulus, axis=1)
                else:
                    # Both are mono
                    tactile_audio[start_sample:start_sample+len(self.tactile_stimulus)] += self.tactile_stimulus
                
                # Log successful injection
                injection_log['tactile'].append({
                    'trial_number': trial_number,
                    'trial_type': trial_type,
                    'time_ms': time_ms,
                    'time_sec': time_ms / 1000,
                    'sample_index': start_sample
                })
            else:
                logging.warning(f"Tactile stimulus at {time_ms}ms (trial {trial_number}) exceeds audio length")
        
        if self.config['debug_mode']:
            logging.info(f"Injected stimuli:")
            logging.info(f"  Looming injections completed: {len(injection_log['looming'])}")
            logging.info(f"  Tactile injections completed: {len(injection_log['tactile'])}")
        
        return looming_audio, tactile_audio, injection_log
    
    def save_audio_files(self, participant_id, looming_audio, tactile_audio, injection_log):
        """
        Save the generated audio files and injection log.
        
        Args:
            participant_id: Participant ID
            looming_audio: Looming audio data
            tactile_audio: Tactile audio data
            injection_log: Dictionary with injection details
            
        Returns:
            Tuple of file paths (looming_path, tactile_path, log_path)
        """
        # Create filenames
        looming_filename = f"participant_{participant_id}_design_looming.wav"
        tactile_filename = f"participant_{participant_id}_design_tactile.wav"
        log_filename = f"participant_{participant_id}_design_stimuli_log.csv"
        
        # Create full paths
        looming_path = os.path.join(self.output_dir, looming_filename)
        tactile_path = os.path.join(self.output_dir, tactile_filename)
        log_path = os.path.join(self.output_dir, log_filename)
        
        # Save audio files
        sf.write(looming_path, looming_audio, self.sample_rate)
        sf.write(tactile_path, tactile_audio, self.sample_rate)
        
        # Save injection log as CSV
        # Combine looming and tactile injections
        all_injections = []
        
        for inject in injection_log['looming']:
            inject['stimulus'] = 'looming'
            all_injections.append(inject)
            
        for inject in injection_log['tactile']:
            inject['stimulus'] = 'tactile'
            # Add stimulus_type field for consistency
            inject['stimulus_type'] = 'tactile'
            all_injections.append(inject)
        
        # Convert to DataFrame and sort by time
        log_df = pd.DataFrame(all_injections)
        if not log_df.empty:
            log_df = log_df.sort_values('time_ms').reset_index(drop=True)
            
            # Save to CSV
            log_df.to_csv(log_path, index=False)
        else:
            # Create empty log file
            with open(log_path, 'w') as f:
                f.write("No stimuli injected\n")
        
        if self.config['debug_mode']:
            logging.info(f"Saved audio files for participant {participant_id}:")
            logging.info(f"  Looming: {looming_path}")
            logging.info(f"  Tactile: {tactile_path}")
            logging.info(f"  Log: {log_path}")
        
        return looming_path, tactile_path, log_path
    
    def verify_audio_sync(self, injection_log):
        """
        Verify audio synchronization by comparing expected SOA values against actual.
        
        Args:
            injection_log: Dictionary with injection details
            
        Returns:
            DataFrame with verification results
        """
        verification_results = []
        
        # Create dict for faster lookup
        looming_dict = {inject['trial_number']: inject for inject in injection_log['looming']}
        tactile_dict = {inject['trial_number']: inject for inject in injection_log['tactile']}
        
        # Check each looming trial that should have a tactile counterpart
        for trial_num, looming in looming_dict.items():
            if looming['trial_type'] == 'catch':
                continue  # Skip catch trials
                
            if trial_num in tactile_dict:
                tactile = tactile_dict[trial_num]
                
                actual_soa_ms = tactile['time_ms'] - looming['time_ms']
                
                verification_results.append({
                    'trial_number': trial_num,
                    'trial_type': looming['trial_type'],
                    'looming_time_ms': looming['time_ms'],
                    'tactile_time_ms': tactile['time_ms'],
                    'actual_soa_ms': actual_soa_ms
                })
        
        results_df = pd.DataFrame(verification_results)
        
        if self.config['debug_mode'] and not results_df.empty:
            # Print first few verification results
            logging.info(f"Verification sample (first 5 trials):")
            
            for _, row in results_df.head(5).iterrows():
                logging.info(f"  Trial {row['trial_number']} ({row['trial_type']}): " 
                            f"SOA = {row['actual_soa_ms']:.2f}ms")
            
            # Calculate statistics
            mean_soa = results_df['actual_soa_ms'].mean()
            min_soa = results_df['actual_soa_ms'].min()
            max_soa = results_df['actual_soa_ms'].max()
            
            logging.info(f"SOA statistics:")
            logging.info(f"  Mean: {mean_soa:.2f}ms")
            logging.info(f"  Min: {min_soa:.2f}ms")
            logging.info(f"  Max: {max_soa:.2f}ms")
        
        return results_df
    
    def process_participant(self, participant_id, empty_audio):
        """
        Process a single participant.
        
        Args:
            participant_id: Participant ID
            empty_audio: Empty audio data to use as base
            
        Returns:
            Tuple of (success, file_paths)
        """
        try:
            if self.config['debug_mode']:
                logging.info(f"\n=== Processing participant {participant_id} ===")
            
            # Load design CSV
            design_df = self.load_participant_design(participant_id)
            
            # Create injection map
            injection_map = self.create_injection_map(design_df)
            
            # Create participant audio
            looming_audio, tactile_audio, injection_log = self.create_participant_audio(
                participant_id, empty_audio, injection_map)
            
            # Verify synchronization
            verification = self.verify_audio_sync(injection_log)
            
            # Save audio files
            looming_path, tactile_path, log_path = self.save_audio_files(
                participant_id, looming_audio, tactile_audio, injection_log)
            
            return True, (looming_path, tactile_path, log_path)
            
        except Exception as e:
            logging.error(f"Error processing participant {participant_id}: {e}")
            import traceback
            traceback.print_exc()
            return False, None
    
    def run(self, participant_ids=None):
        """
        Run audio generation for multiple participants.
        
        Args:
            participant_ids: List of participant IDs or None for all
            
        Returns:
            Dictionary with results
        """
        # If no participant IDs provided, find all design CSVs
        if participant_ids is None:
            # Find all participant design files
            design_dir = self.config['paths']['design_dir']
            design_files = [f for f in os.listdir(design_dir) if f.startswith('participant_') and f.endswith('_design.csv')]
            
            # Extract participant IDs
            participant_ids = []
            for filename in design_files:
                match = re.match(r'participant_(\d+)_design\.csv', filename)
                if match:
                    participant_ids.append(int(match.group(1)))
            
            participant_ids.sort()  # Sort numerically
        
        if self.config['debug_mode']:
            logging.info(f"=== Starting PPSAudioGenerator ===")
            logging.info(f"Processing {len(participant_ids)} participants: {participant_ids}")
        
        # Find the maximum trial number across all CSVs
        max_trial = self.find_max_trial_number()
        
        # Create empty audio once, for use with all participants
        empty_audio = self.get_empty_audio(max_trial)
        
        results = {}
        for pid in participant_ids:
            success, paths = self.process_participant(pid, empty_audio)
            results[pid] = {'success': success, 'paths': paths}
        
        # Summary
        if self.config['debug_mode']:
            success_count = sum(1 for info in results.values() if info['success'])
            logging.info(f"\n=== GENERATION SUMMARY ===")
            logging.info(f"Successfully generated audio for {success_count}/{len(results)} participants.")
        
        return results

# ----------------------------------------------------------------------
# MAIN FUNCTION
# ----------------------------------------------------------------------
def main():
    """Main function to run the audio generator."""
    # Print configuration summary
    print("\n===== PPS EXPERIMENT AUDIO GENERATOR =====")
    print(f"Intro file: {AUDIO_CONFIG['paths']['intro']}")
    print(f"Repetition segment file: {AUDIO_CONFIG['paths']['repetition_segment']}")
    print(f"Outro file: {AUDIO_CONFIG['paths']['outro']}")
    print(f"Output directory: {AUDIO_CONFIG['paths']['output_dir']}")
    
    # Create the generator
    generator = PPSAudioGenerator(AUDIO_CONFIG)
    
    # Get participant IDs to process
    # You can specify specific IDs here or let it find all design files
    participant_ids = None  # Process all available participants
    # participant_ids = [0, 1, 2]  # Process specific participants
    
    # Run the generator
    results = generator.run(participant_ids)
    
    # Print summary
    print("\n===== GENERATION SUMMARY =====")
    success_count = sum(1 for info in results.values() if info['success'])
    print(f"Successfully generated audio for {success_count}/{len(results)} participants.")
    
    for pid, info in results.items():
        if info['success']:
            looming_path, tactile_path, log_path = info['paths']
            looming_file = os.path.basename(looming_path)
            tactile_file = os.path.basename(tactile_path)
            print(f"Participant {pid}: Files generated")
            print(f"  Looming: {looming_file}")
            print(f"  Tactile: {tactile_file}")
        else:
            print(f"Participant {pid}: ERROR - audio generation failed.")

if __name__ == "__main__":
    main()


===== PPS EXPERIMENT AUDIO GENERATOR =====
Intro file: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_intro.wav
Repetition segment file: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_repetitionsegment.wav
Outro file: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\BoxBreathing_outro.wav
Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio


2025-03-27 17:56:15,882 - INFO - Loaded looming stimulus 'blue' from frontal_looming_blue_noise.wav
2025-03-27 17:56:15,891 - INFO - Loaded looming stimulus 'brown' from frontal_looming_brown_noise.wav
2025-03-27 17:56:15,896 - INFO - Loaded looming stimulus 'pink' from frontal_looming_pink_noise.wav
2025-03-27 17:56:15,899 - INFO - Loaded looming stimulus 'white' from frontal_looming_white_noise.wav
2025-03-27 17:56:15,903 - INFO - Loaded audio files:
2025-03-27 17:56:15,905 - INFO -   Intro: 312.49 seconds, (14999552, 2)
2025-03-27 17:56:15,906 - INFO -   Repetition Segment: 272.55 seconds, (13082624, 2)
2025-03-27 17:56:15,906 - INFO -   Outro: 100.40 seconds, (4818976, 2)
2025-03-27 17:56:15,906 - INFO -   Sample Rate: 48000 Hz
2025-03-27 17:56:15,908 - INFO -   Looming 'blue': 2.76 seconds, (132300, 2)
2025-03-27 17:56:15,908 - INFO -   Looming 'brown': 2.76 seconds, (132300, 2)
2025-03-27 17:56:15,911 - INFO -   Looming 'pink': 2.76 seconds, (132300, 2)
2025-03-27 17:56:15,911 - 


===== GENERATION SUMMARY =====
Successfully generated audio for 100/100 participants.
Participant 0: Files generated
  Looming: participant_0_design_looming.wav
  Tactile: participant_0_design_tactile.wav
Participant 1: Files generated
  Looming: participant_1_design_looming.wav
  Tactile: participant_1_design_tactile.wav
Participant 2: Files generated
  Looming: participant_2_design_looming.wav
  Tactile: participant_2_design_tactile.wav
Participant 3: Files generated
  Looming: participant_3_design_looming.wav
  Tactile: participant_3_design_tactile.wav
Participant 4: Files generated
  Looming: participant_4_design_looming.wav
  Tactile: participant_4_design_tactile.wav
Participant 5: Files generated
  Looming: participant_5_design_looming.wav
  Tactile: participant_5_design_tactile.wav
Participant 6: Files generated
  Looming: participant_6_design_looming.wav
  Tactile: participant_6_design_tactile.wav
Participant 7: Files generated
  Looming: participant_7_design_looming.wav
  Tac